# Jupyphant Enhanced Neo Tree Test Notebook

This notebook contains automated tests for the Jupyphant's Neo object tree.

In [ ]:
import unittest
import neo
import quantities as pq
import numpy as np
from jupyphant.jupyphant import Jupyphant
from elephant import statistics
import __main__

In [ ]:
class TestNeoTree(unittest.TestCase):
    
    def setUp(self):
        self.jupyphant = Jupyphant()
        self.jupyphant.create_tree()
        self.vars_to_cleanup = []
        
    def tearDown(self):
        for var in self.vars_to_cleanup:
            if hasattr(__main__, var):
                delattr(__main__, var)
    
    def _add_to_main(self, name, obj):
        setattr(__main__, name, obj)
        self.vars_to_cleanup.append(name)

    def test_add_and_delete_block(self):
        # Create a new Block
        my_block = neo.Block(name="Test Block")
        self._add_to_main("my_block", my_block)
        
        # Update the tree and check if the block is there
        self.jupyphant.update_tree()
        self.assertIn("my_block", self.jupyphant.neo_objs_and_lists_of_neo_objs_with_var_name)
        
        # Delete the block
        del my_block
        delattr(__main__, 'my_block')
        
        # Update the tree and check if the block is gone
        self.jupyphant.update_tree()
        self.assertNotIn("my_block", self.jupyphant.neo_objs_and_lists_of_neo_objs_with_var_name)

    def test_modify_block_name(self):
        # Create a new Block
        my_block = neo.Block(name="Initial Name")
        self._add_to_main("my_block", my_block)
        
        # Update the tree
        self.jupyphant.update_tree()
        self.assertEqual(self.jupyphant.neo_objs_and_lists_of_neo_objs_with_var_name["my_block"].name, "Initial Name")
        
        # Modify the block name
        my_block.name = "New Name"
        
        # Update the tree and check if the name has changed
        self.jupyphant.update_tree()
        self.assertEqual(self.jupyphant.neo_objs_and_lists_of_neo_objs_with_var_name["my_block"].name, "New Name")

    def test_modify_block_description(self):
        # Create a new Block
        my_block = neo.Block(name="Test Block", description="Initial description")
        self._add_to_main("my_block", my_block)
        
        # Update the tree
        self.jupyphant.update_tree()
        self.assertEqual(self.jupyphant.neo_objs_and_lists_of_neo_objs_with_var_name["my_block"].description, "Initial description")
        
        # Modify the block description
        my_block.description = "New description"
        
        # Update the tree and check if the description has changed
        self.jupyphant.update_tree()
        self.assertEqual(self.jupyphant.neo_objs_and_lists_of_neo_objs_with_var_name["my_block"].description, "New description")

    def test_add_and_remove_segment(self):
        # Create a block and a segment
        my_block = neo.Block(name="Test Block")
        self._add_to_main("my_block", my_block)
        my_segment = neo.Segment(name="Test Segment")
        self._add_to_main("my_segment", my_segment)
        my_block.segments.append(my_segment)
        
        # Update the tree
        self.jupyphant.update_tree()
        block_from_jupyphant = self.jupyphant.neo_objs_and_lists_of_neo_objs_with_var_name["my_block"]
        self.assertEqual(len(block_from_jupyphant.segments), 1)
        self.assertEqual(block_from_jupyphant.segments[0].name, "Test Segment")
        
        # Remove the segment
        my_block.segments.remove(my_segment)
        
        # Update the tree
        self.jupyphant.update_tree()
        self.assertEqual(len(block_from_jupyphant.segments), 0)

    def test_add_analog_signal_to_segment(self):
        # Create a block, a segment and an analog signal
        my_block = neo.Block(name="Test Block")
        self._add_to_main("my_block", my_block)
        my_segment = neo.Segment(name="Test Segment")
        self._add_to_main("my_segment", my_segment)
        my_block.segments.append(my_segment)
        anasig = neo.AnalogSignal(np.random.randn(100, 1) * pq.mV, sampling_rate=10 * pq.Hz)
        self._add_to_main("anasig", anasig)
        my_segment.analogsignals.append(anasig)
        
        # Update the tree
        self.jupyphant.update_tree()
        block_from_jupyphant = self.jupyphant.neo_objs_and_lists_of_neo_objs_with_var_name["my_block"]
        segment_from_jupyphant = block_from_jupyphant.segments[0]
        self.assertEqual(len(segment_from_jupyphant.analogsignals), 1)

    def test_add_spiketrain_to_segment(self):
        # Create a block, a segment and a spiketrain
        my_block = neo.Block(name="Test Block")
        self._add_to_main("my_block", my_block)
        my_segment = neo.Segment(name="Test Segment")
        self._add_to_main("my_segment", my_segment)
        my_block.segments.append(my_segment)
        st = neo.SpikeTrain(np.arange(0, 10) * pq.s, t_stop=10 * pq.s)
        self._add_to_main("st", st)
        my_segment.spiketrains.append(st)
        
        # Update the tree
        self.jupyphant.update_tree()
        block_from_jupyphant = self.jupyphant.neo_objs_and_lists_of_neo_objs_with_var_name["my_block"]
        segment_from_jupyphant = block_from_jupyphant.segments[0]
        self.assertEqual(len(segment_from_jupyphant.spiketrains), 1)

    def test_annotations(self):
        # Create a block with an annotation
        my_block = neo.Block(name="Test Block")
        self._add_to_main("my_block", my_block)
        my_block.annotations['my_annotation'] = 'my_value'
        
        # Update the tree
        self.jupyphant.update_tree()
        block_from_jupyphant = self.jupyphant.neo_objs_and_lists_of_neo_objs_with_var_name["my_block"]
        self.assertEqual(block_from_jupyphant.annotations['my_annotation'], 'my_value')
        
        # Modify the annotation
        my_block.annotations['my_annotation'] = 'new_value'
        self.jupyphant.update_tree()
        self.assertEqual(block_from_jupyphant.annotations['my_annotation'], 'new_value')

    def test_group(self):
        # Create a block and a group
        my_block = neo.Block(name="Test Block")
        self._add_to_main("my_block", my_block)
        my_group = neo.Group(name="Test Group")
        self._add_to_main("my_group", my_group)
        my_block.groups.append(my_group)
        
        # Update the tree
        self.jupyphant.update_tree()
        block_from_jupyphant = self.jupyphant.neo_objs_and_lists_of_neo_objs_with_var_name["my_block"]
        self.assertEqual(len(block_from_jupyphant.groups), 1)
        self.assertEqual(block_from_jupyphant.groups[0].name, "Test Group")

    def test_firing_rate(self):
        # Create a spiketrain
        st = neo.SpikeTrain(np.arange(0, 10) * pq.s, t_stop=10 * pq.s)
        self._add_to_main("st", st)
        
        # Calculate firing rate
        rate = statistics.mean_firing_rate(st)
        self.assertAlmostEqual(rate.magnitude, 1.0, places=1)

    def test_epoch_and_event(self):
        # Create a block and a segment
        my_block = neo.Block(name="Test Block")
        self._add_to_main("my_block", my_block)
        my_segment = neo.Segment(name="Test Segment")
        self._add_to_main("my_segment", my_segment)
        my_block.segments.append(my_segment)
        
        # Create an epoch and an event
        ep = neo.Epoch(times=np.arange(0, 10, 2) * pq.s, durations=0.5 * pq.s, labels=np.array([f'epoch{i}' for i in range(5)]))
        self._add_to_main("ep", ep)
        ev = neo.Event(times=np.arange(1, 11, 2) * pq.s, labels=np.array([f'event{i}' for i in range(5)]))
        self._add_to_main("ev", ev)
        my_segment.epochs.append(ep)
        my_segment.events.append(ev)
        
        # Update the tree
        self.jupyphant.update_tree()
        block_from_jupyphant = self.jupyphant.neo_objs_and_lists_of_neo_objs_with_var_name["my_block"]
        segment_from_jupyphant = block_from_jupyphant.segments[0]
        self.assertEqual(len(segment_from_jupyphant.epochs), 1)
        self.assertEqual(len(segment_from_jupyphant.events), 1)

    def test_array_annotations(self):
        # Create a spiketrain with array annotations
        st = neo.SpikeTrain(np.arange(0, 10) * pq.s, t_stop=10 * pq.s)
        self._add_to_main("st", st)
        st.array_annotations['my_array'] = np.arange(10)
        
        # Update the tree
        self.jupyphant.update_tree()
        st_from_jupyphant = self.jupyphant.neo_objs_and_lists_of_neo_objs_with_var_name["st"]
        self.assertIn('my_array', st_from_jupyphant.array_annotations)
        self.assertTrue(np.array_equal(st_from_jupyphant.array_annotations['my_array'], np.arange(10)))


In [ ]:
def run_tests():
    suite = unittest.TestSuite()
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(TestNeoTree)
    runner = unittest.TextTestRunner()
    result = runner.run(suite)
    return result

In [ ]:
result = run_tests()
result